# 03. Feature Engineering and Data Splitting

This notebook creates timestamp features, removes obvious leakage, and performs the 70/15/15 train-calibration-test split required for conformal prediction.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.append(str(Path.cwd().parent))
from src.upi_fraud_pipeline import clean_dataframe, infer_target_column, prepare_features, split_train_calibration_test

df = pd.read_csv(Path.cwd().parent / 'data' / 'processed' / 'cleaned_upi_dataset.csv')
target_col = infer_target_column(df)
df = clean_dataframe(df, target_column=target_col)
X, y, numeric_cols, categorical_cols = prepare_features(df, target_col)
print('X shape:', X.shape)
print('Numeric columns:', numeric_cols[:10])
print('Categorical columns:', categorical_cols[:10])

In [ ]:
leakage_cols = [c for c in X.columns if any(token in c.lower() for token in ['id', 'account', 'customer', 'session'])]
print('Leakage candidates:', leakage_cols)
X = X.drop(columns=leakage_cols, errors='ignore')

In [ ]:
X_train, X_cal, X_test, y_train, y_cal, y_test = split_train_calibration_test(X, y)
print('Train:', X_train.shape, y_train.shape)
print('Calibration:', X_cal.shape, y_cal.shape)
print('Test:', X_test.shape, y_test.shape)
print('Fraud rate, train:', y_train.mean())
print('Fraud rate, calibration:', y_cal.mean())
print('Fraud rate, test:', y_test.mean())